# 05 — Delta medallion: MovieLens + Wikipedia

Builds a 3-layer Delta lakehouse from the raw open data ingested by **notebook 04**:

- **Bronze** — raw, schema-on-read, untouched. One Delta table per source file.
- **Silver** — typed, cleaned, denormalised. Genres parsed into arrays, timestamps cast, JSON exploded.
- **Gold** — business questions: top-rated movies, KPIs per genre, daily Wikipedia leaders.

Also demonstrates `MERGE INTO` for incrementals and `DESCRIBE HISTORY` for audit.

In [ ]:
from spark_session import get_spark
from transforms import explode_pageviews
from pyspark.sql import functions as F, types as T

spark = get_spark("05-delta-medallion")
RAW = "s3a://raw-data"
WH  = "s3a://spark-warehouse/delta/movielens"

## Bronze — raw → Delta, no transformations

In [ ]:
for name in ("movies", "ratings", "tags", "links"):
    (spark.read.option("header", True).csv(f"{RAW}/movielens/{name}.csv")
         .write.format("delta").mode("overwrite")
         .save(f"{WH}/bronze/{name}"))
    print(f"  bronze/{name} ok")

(spark.read.json(f"{RAW}/wikipedia/*.json")
      .write.format("delta").mode("overwrite")
      .save("s3a://spark-warehouse/delta/wikipedia/bronze/pageviews"))
print("  bronze/wikipedia/pageviews ok")

In [ ]:
spark.read.format("delta").load(f"{WH}/bronze/movies").show(5, truncate=False)
spark.read.format("delta").load("s3a://spark-warehouse/delta/wikipedia/bronze/pageviews").printSchema()

## Silver — types, parsed structures, no nulls in keys

In [ ]:
# movies: split "Adventure|Sci-Fi" → array, extract year from "Toy Story (1995)"
movies = (spark.read.format("delta").load(f"{WH}/bronze/movies")
    .withColumn("movieId", F.col("movieId").cast("int"))
    .withColumn("year",   F.regexp_extract("title", r"\((\d{4})\)$", 1).cast("int"))
    .withColumn("title_clean", F.regexp_replace("title", r"\s*\(\d{4}\)$", ""))
    .withColumn("genres", F.split("genres", r"\|"))
    .where(F.col("movieId").isNotNull())
    .select("movieId", "title_clean", "year", "genres"))
movies.write.format("delta").mode("overwrite").save(f"{WH}/silver/movies")

# ratings: timestamp → real timestamp, rating → double
ratings = (spark.read.format("delta").load(f"{WH}/bronze/ratings")
    .select(
        F.col("userId").cast("int").alias("user_id"),
        F.col("movieId").cast("int").alias("movie_id"),
        F.col("rating").cast("double").alias("rating"),
        F.col("timestamp").cast("long").alias("ts_epoch"),
        F.from_unixtime("timestamp").cast("timestamp").alias("rated_at"),
    )
    .where("user_id IS NOT NULL AND movie_id IS NOT NULL"))
ratings.write.format("delta").mode("overwrite").save(f"{WH}/silver/ratings")

# pageviews: explode articles[] into one row per (date, article) — shared with notebook 06
pv_raw = spark.read.format("delta").load("s3a://spark-warehouse/delta/wikipedia/bronze/pageviews")
pv = explode_pageviews(pv_raw)
(pv.write.format("delta").mode("overwrite").partitionBy("day")
      .save("s3a://spark-warehouse/delta/wikipedia/silver/pageviews"))

# Read the counts back from the written tables, so the numbers verify the writes themselves.
for label, path in (
    ("movies",    f"{WH}/silver/movies"),
    ("ratings",   f"{WH}/silver/ratings"),
    ("pageviews", "s3a://spark-warehouse/delta/wikipedia/silver/pageviews"),
):
    print(f"silver/{label:<10} {spark.read.format('delta').load(path).count():>7,}")

## Gold — business questions

In [ ]:
# Gold 1: top 20 movies with ≥50 ratings, ranked by mean rating
movies = spark.read.format("delta").load(f"{WH}/silver/movies")
ratings = spark.read.format("delta").load(f"{WH}/silver/ratings")

movie_kpis = (ratings.groupBy("movie_id")
    .agg(F.avg("rating").alias("avg_rating"),
         F.count("*").alias("n_ratings"))
    .where("n_ratings >= 50")
    .join(movies, F.col("movie_id") == movies.movieId)
    .select("movie_id", "title_clean", "year", "genres", "n_ratings",
            F.round("avg_rating", 3).alias("avg_rating")))
(movie_kpis.write.format("delta").mode("overwrite").save(f"{WH}/gold/movie_kpis"))

(movie_kpis.orderBy(F.desc("avg_rating"), F.desc("n_ratings")).limit(20)
          .show(truncate=False))

In [ ]:
# Gold 2: rating distribution per genre (genres array exploded)
genre_kpis = (movie_kpis.select(F.explode("genres").alias("genre"), "avg_rating", "n_ratings")
    .groupBy("genre")
    .agg(F.count("*").alias("n_movies"),
         F.round(F.avg("avg_rating"), 3).alias("genre_avg_rating"),
         F.sum("n_ratings").alias("total_ratings"))
    .orderBy(F.desc("genre_avg_rating")))
(genre_kpis.write.format("delta").mode("overwrite").save(f"{WH}/gold/genre_kpis"))
genre_kpis.show(truncate=False)

In [ ]:
# Gold 3: daily Wikipedia top-1 article (the most-viewed page on each day in the window)
from pyspark.sql import Window

pv = spark.read.format("delta").load("s3a://spark-warehouse/delta/wikipedia/silver/pageviews")
w = Window.partitionBy("day").orderBy(F.col("views").desc())
daily_top = (pv.withColumn("rn", F.row_number().over(w))
               .where("rn = 1")
               .select("day", "article", "views")
               .orderBy("day"))
daily_top.write.format("delta").mode("overwrite").save("s3a://spark-warehouse/delta/wikipedia/gold/daily_top")
daily_top.show(truncate=False)

## MERGE — simulate an incremental update

Pretend new tags arrived. Upsert into the silver tags table using Delta `MERGE INTO`.

> **Caveat:** `(user_id, movie_id)` is *not* unique in MovieLens tags — a user can apply several tags to the same film, so the `WHEN MATCHED` clause overwrites **all** of that user's tags for the film. A real pipeline would include `tag` in the merge key, or deduplicate the source batch first (`MERGE` errors out if several *source* rows match one target row).

In [ ]:
# First materialise silver/tags so we have a target to merge into
tags = (spark.read.format("delta").load(f"{WH}/bronze/tags")
    .select(
        F.col("userId").cast("int").alias("user_id"),
        F.col("movieId").cast("int").alias("movie_id"),
        F.col("tag").alias("tag"),
        F.from_unixtime("timestamp").cast("timestamp").alias("tagged_at"),
    ))
tags.write.format("delta").mode("overwrite").save(f"{WH}/silver/tags")

spark.sql(f"CREATE OR REPLACE TEMP VIEW silver_tags AS SELECT * FROM delta.`{WH}/silver/tags`")

# Hand-rolled update batch: one new tag, one corrected tag
updates = spark.createDataFrame(
    [(99999, 1, "hidden gem", "2026-06-03 12:00:00"),       # brand-new
     (2,     60756, "comedy gold", "2026-06-03 12:01:00")], # would replace an existing (user_id=2, movie_id=60756) tag
    "user_id INT, movie_id INT, tag STRING, tagged_at_str STRING"
).withColumn("tagged_at", F.to_timestamp("tagged_at_str")).drop("tagged_at_str")
updates.createOrReplaceTempView("updates")

spark.sql(f"""
MERGE INTO delta.`{WH}/silver/tags` t
USING updates u
ON  t.user_id = u.user_id AND t.movie_id = u.movie_id
WHEN MATCHED THEN UPDATE SET tag = u.tag, tagged_at = u.tagged_at
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql(f"SELECT * FROM delta.`{WH}/silver/tags` WHERE user_id IN (99999, 2) ORDER BY user_id, tagged_at DESC").show(5, truncate=False)

## Audit — DESCRIBE HISTORY

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{WH}/silver/tags`").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

# Drop the SparkSession so the History Server can ingest this app's event log
spark.stop()